In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
import json
from pathlib import Path

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from utils.tools import aggregate_results

In [ ]:
def get_suite(row):

    n_demos = row["number of demonstrations"]
    type_demos = row["type of demonstrations"][0:3]
    instr = "impl" if row["use instructions"] == "no" else "expl"

    return f"{n_demos}-{type_demos}-{instr}"

def capitalize(s):
    return s[0].upper() + s[1:]

def order_suites(strings):
    return sorted(strings, key=lambda s: (
        s.split('-')[2],  # implicit-excplict
        s.split('-')[0],  # num demos
        s.split('-')[1],  # demo type
    ))

def order_models(strings):
    return sorted(strings, key=lambda s: (
        int(s.split('-')[1][:-1]),  # Param count, B dropped
        #s.split('-')[0],  # Model name
    ), reverse=True)

def bold_extreme_values(s, by_model=True):
    # Bold max for mean

    if not by_model:
        is_max = s == s.max()
        return ['font-weight: bold' if v else '' for v in is_max]
    
    font_array = []

    model_level = s.index.names.index('Model')
    models = s.index.get_level_values(model_level)
    models = pd.Series(list(models)).unique()

    idx = pd.IndexSlice
    
    for model in models:   
        values_by_model = s.loc[idx[model]]
        is_max = values_by_model == values_by_model.max()
        font_array += ['font-weight: bold' if v else '' for v in is_max]

    return font_array

In [ ]:
def heatmap(df, metric_cols):
    fig, axes = plt.subplots(len(metric_cols), 1, figsize=(10, 15), sharex=True, sharey=True)

    for i, ax in enumerate(axes):

        label, metric, *_ = metric_cols[i].split(" ")

        pivoted = df.pivot(index='model', columns='suite', values=metric_cols[i])
        pivoted = pivoted.reindex(order_suites(pivoted.columns), axis=1)
        pivoted = pivoted.reindex(order_models(pivoted.index), axis=0)
        
        ax.set_title(capitalize(label))
        
        sns.heatmap(
            pivoted,
            fmt=".2g",
            annot=True,
            square=True,
            cmap="Blues",
            ax=ax,
            cbar=False
        )

        if i == 2:
            # Rotate xticks
            ax.set_xticklabels(ax.get_xticklabels(), rotation=40)

    mappable = ax.get_children()[0]
    plt.colorbar(mappable, ax=axes, orientation = 'vertical')

    return fig, ax

In [ ]:
def my_heatmap(df, metric_cols):
    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True, sharey=True)

    plot_ax = [axes[0, 0], axes[0, 1], axes[1, 0]]

    for i, ax in enumerate(plot_ax):

        label, metric, *_ = metric_cols[i].split(" ")

        pivoted = df.pivot(index='model', columns='suite', values=metric_cols[i])
        pivoted = pivoted.reindex(order_suites(pivoted.columns), axis=1)
        pivoted = pivoted.reindex(order_models(pivoted.index), axis=0)
        
        ax.set_title(capitalize(label))
        
        sns.heatmap(
            pivoted,
            fmt=".2g",
            annot=True,
            square=True,
            cmap="Blues",
            ax=ax,
            cbar=False
        )

        if i == 2:
            # Rotate xticks
            ax.set_xticklabels(ax.get_xticklabels(), rotation=40)

    # Move plots
    topright = plot_ax[1]
    pos = topright.get_position()
    pos.x0 = pos.x0 - 0.05
    pos.x1 = pos.x1 - 0.05
    topright.set_position(pos)

    bottomleft = plot_ax[2]
    pos = bottomleft.get_position()
    pos.x0 = pos.x0 + 0.185
    pos.x1 = pos.x1 + 0.185
    bottomleft.set_position(pos)

    plt.axis("off")

    mappable = ax.get_children()[0]
    #plt.colorbar(mappable, ax=axes, orientation = 'horizontal')

    return fig, ax

In [ ]:
metric_cols_f1 = ["theme f1 mean", "topic f1 mean", "concept f1 mean"]
metric_cols_prec = ["theme precision mean", "topic precision mean", "concept precision mean"]
metric_cols_rec = ["theme recall mean", "topic recall mean", "concept recall mean"]

###
# HEATMAP F1
###
res = pd.read_csv("./data/metrics.csv", sep=";")
res = res.sort_values(by=["use instructions", "number of demonstrations", "type of demonstrations"])
res["suite"] = res.apply(get_suite, axis=1)

fig, ax = heatmap(res, metric_cols_f1)
#plt.savefig("./latex/images/f1_per_suite_v3", bbox_inches="tight")

In [ ]:
fig, ax = my_heatmap(res, metric_cols_prec)
#plt.savefig("./latex/images/precision_per_suite_v2", bbox_inches="tight")

In [ ]:
fig, ax = my_heatmap(res, metric_cols_rec)
#plt.savefig("./latex/images/recall_per_suite_v2", bbox_inches="tight")